# 05. 모델 비교

동일한 변수와 시간 분할에서 LightGBM과 CatBoost를 비교합니다. 모델 선택은 RMSLE를 우선하고 MAE와 운영 복잡도를 함께 고려합니다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_PATH = PROJECT_ROOT / "outputs"

In [ ]:
def load_metrics(path, model_name):
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as file:
        metrics = json.load(file)
    return {"model": model_name, **metrics}


def available_results(output_path):
    lightgbm_path = output_path / "metrics_lightgbm.json"
    if not lightgbm_path.exists():
        lightgbm_path = output_path / "metrics.json"
    candidates = [
        load_metrics(lightgbm_path, "LightGBM"),
        load_metrics(output_path / "metrics_catboost.json", "CatBoost"),
    ]
    return pd.DataFrame([result for result in candidates if result is not None])

## 비교 조건

| 항목 | 공통 조건 |
|---|---|
| 예측 목표 | 점포·상품군별 1~7일 뒤 일별 판매량 |
| 모델 구조 | `forecast_horizon`을 입력받는 통합 글로벌 모델 |
| 학습·검증 분리 | 최근 28개 예측 기준일 검증, 이후 정답 학습 제외 |
| 입력 변수 | 동일한 lag·이동평균·달력·점포·상품군 변수 |
| 평가지표 | RMSLE 우선, MAE 보조 |
| 조기 종료 | 검증 성능이 개선되지 않으면 중단 |

In [ ]:
results = available_results(OUTPUT_PATH)
comparison_columns = [
    "model", "rmsle", "mae", "previous_week_rmsle",
    "previous_week_mae", "same_weekday_4w_rmsle",
    "same_weekday_4w_mae", "train_rows", "validation_rows",
]
results[[column for column in comparison_columns if column in results.columns]].set_index("model").round(4)

In [ ]:
if len(results) > 0:
    plot_data = results.set_index("model")[["rmsle", "mae"]]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plot_data["rmsle"].sort_values().plot.barh(ax=axes[0], title="RMSLE")
    plot_data["mae"].sort_values().plot.barh(ax=axes[1], color="tab:orange", title="MAE")
    axes[0].set(xlabel="lower is better", ylabel="")
    axes[1].set(xlabel="lower is better", ylabel="")
    plt.tight_layout()

## CatBoost 실행

아래 명령으로 동일한 데이터 분할에서 CatBoost를 학습한 뒤 이 노트북을 다시 실행합니다.

```powershell
.venv\Scripts\python.exe scripts\run_experiment.py --config configs\catboost.toml
```

초안 단계에서는 LightGBM 결과만 표시될 수 있습니다. CatBoost 결과가 생성되면 표와 그래프에 자동으로 추가됩니다.

## 선택 기준

- RMSLE가 낮은 모델을 우선합니다.
- MAE가 크게 악화되지 않는지 함께 확인합니다.
- 상품군·점포별 오차가 안정적인지 `06_error_analysis`에서 확인합니다.
- 성능 차이가 작다면 학습 속도와 설명 편의성이 높은 모델을 선택합니다.